In [ ]:
!pip install groq

In [ ]:
import json
import requests
from google.colab import userdata
from groq import Groq


In [ ]:
client = Groq(
  api_key=userdata.get('GROQ_API_KEY')
)

In [ ]:
def get_weather(location):
    api_key = userdata.get('WEATHER_API_KEY')



    url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&units=metric&appid={api_key}"

    response = requests.get(url)
    data = response.json()


    if data.get("cod") == 200:
        return json.dumps({
            "location": location,
            "temperature": data["main"]["temp"],
            "description": data["weather"][0]["description"]
        })
    else:
        return json.dumps({
            "error": "Oops! Something went wrong."
        })

print(get_weather("Hyderabad"))

{"location": "Hyderabad", "temperature": 33.23, "description": "clear sky"}


In [ ]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current weather for a city",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "City name like Mumbai, London"
          }},
        "required": ["location"]
      }}}
]

In [ ]:
llm_messages = [
  {
    "role": "system",
    "content": "You are a weather assistant. Use get_weather function when asked about weather."
  },]

In [ ]:
response = client.chat.completions.create(
  model="openai/gpt-oss-120b",
  messages=llm_messages,
  tools=tools,
  tool_choice="auto"
)

response_message = response.choices[0].message


if response_message.tool_calls:
  tool_call = response_message.tool_calls[0]
  arguments = json.loads(tool_call.function.arguments)
  location = arguments['location']
  weather_data = get_weather(location)

  llm_messages.append(response_message)

  llm_messages.append({
      "role": "tool",
      "tool_call_id": tool_call.id,
      "content": json.dumps(weather_data)
  })

  final_response = client.chat.completions.create(
      messages = llm_messages,
      model = "openai/gpt-oss-120b",
      tools = tools,
      tool_choice = "auto"
  )

  print(final_response.choices[0].message.content)